# Code Quality Loop (Calculator)

A production-style loop where a coder writes a calculator module, a reviewer scores the code (PEP8-ish), a recommender proposes fixes, and the coder applies them until the quality score ≥ 9.5. A documentation agent runs alongside every step to capture running notes.

In [ ]:
# Runtime setup: optional OpenAI key, fallback to stub LLM
import os
from getpass import getpass

if not os.getenv('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass('Enter OPENAI_API_KEY (press Enter to skip): ')

from agentic_codex import AgentBuilder, Context, FunctionAdapter, EnvOpenAIAdapter
from agentic_codex.core.schemas import AgentStep, Message

def stub_llm(prompt: str) -> str:
    lines = [ln.strip() for ln in prompt.splitlines() if ln.strip()]
    return '[stub llm] ' + (lines[-1][:160] if lines else 'response')

try:
    llm = EnvOpenAIAdapter(model='gpt-4o-mini')
except Exception:
    llm = FunctionAdapter(stub_llm)


## Agent definitions
- **coder**: writes/improves calculator code using goal and prior code.
- **reviewer**: scores code (0–10) and reports issues in JSON.
- **recommender**: suggests concrete fixes.
- **doc_agent**: documents the evolving code, review, and recommendations after each step.

In [ ]:
def make_coder():
    def step(ctx: Context) -> AgentStep:
        code = ctx.scratch.get('code', '')
        goal = ctx.goal
        prompt = f"Write or improve a Python calculator module. Goal: {goal}. Current code:\n{code}"
        new_code = ctx.llm.generate(prompt) if ctx.llm else code
        return AgentStep(out_messages=[Message(role='coder', content=new_code)], state_updates={'code': new_code})
    return AgentBuilder(name='coder', role='developer').with_llm(llm).with_step(step).build()

def make_reviewer():
    def step(ctx: Context) -> AgentStep:
        code = ctx.scratch.get('code', '')
        prompt = (
            'You are a PEP8 reviewer. Score from 0-10 (float) and list issues. '
            f"Code:\n{code}\nRespond as JSON with keys 'score' and 'issues'."
        )
        review = ctx.llm.generate(prompt) if ctx.llm else '{"score":7.0,"issues":["stub"]}'
        ctx.scratch['review'] = review
        try:
            # Try to extract JSON inside ```json ... ``` fences
            match = re.search(r"```json\s*(\{.*?\})\s*```", review, re.DOTALL | re.IGNORECASE)
            if match:
                json_str = match.group(1)
            else:
                # If there is no fenced block, assume the whole string is JSON
                json_str = review

            parsed = json.loads(json_str)
            print("parsed review:\n", parsed)

            score = float(parsed.get("score", 0))
        except Exception as e:
            print("Failed to parse review as JSON:", e)
            score = 0.0
        return AgentStep(out_messages=[Message(role='reviewer', content=review)], state_updates={'score': score})
    return AgentBuilder(name='reviewer', role='qa').with_llm(llm).with_step(step).build()

def make_recommender():
    def step(ctx: Context) -> AgentStep:
        code = ctx.scratch.get('code', '')
        review = ctx.scratch.get('review', '')
        prompt = f"Suggest concrete fixes to reach score 10. Code:\n{code}\nReview: {review}"
        recs = ctx.llm.generate(prompt) if ctx.llm else 'Improve formatting and add docstrings.'
        return AgentStep(out_messages=[Message(role='recommender', content=recs)], state_updates={'recs': recs})
    return AgentBuilder(name='recommender', role='advisor').with_llm(llm).with_step(step).build()

def make_doc_agent():
    def step(ctx: Context) -> AgentStep:
        code = ctx.scratch.get('code', '')
        review = ctx.scratch.get('review', '')
        recs = ctx.scratch.get('recs', '')
        prompt = f"Document the calculator code, its review, and recommended fixes. Code:\n{code}\nReview:{review}\nRecs:{recs}"
        doc = ctx.llm.generate(prompt) if ctx.llm else 'Documentation stub.'
        return AgentStep(out_messages=[Message(role='doc', content=doc)], state_updates={'docs': doc})
    return AgentBuilder(name='doc', role='documentation').with_llm(llm).with_step(step).build()

coder = make_coder()
reviewer = make_reviewer()
recommender = make_recommender()
doc_agent = make_doc_agent()


## Run the quality-improvement loop
- Iteratively run coder → reviewer → recommender.
- After *each* agent action, the documentation agent runs to keep notes in near-parallel.
- The coder appends applied fixes.
- Stop when score ≥ 9.5 or after 8 iterations.

In [ ]:
ctx = Context(goal='Build a Python calculator with add/subtract/multiply/divide', scratch={'score': 0.0})
history = []
iteration = 0
while ctx.scratch.get('score', 0.0) < 9.5 and iteration < 8:
    for agent in (coder, reviewer, recommender):
        result = agent.run(ctx)
        history.append((agent.name, result.out_messages[-1].content if result.out_messages else ''))
        ctx.scratch.update(result.state_updates)
        # Documentation agent runs after every agent to capture evolving state
        doc_result = doc_agent.run(ctx)
        ctx.scratch.update(doc_result.state_updates)
        history.append((doc_agent.name, doc_result.out_messages[-1].content if doc_result.out_messages else ''))
        if agent.name == 'coder' and 'recs' in ctx.scratch:
            ctx.scratch['code'] = ctx.scratch.get('code', '') + '\n# Applied fixes: ' + str(ctx.scratch.get('recs'))
    iteration += 1

summary = {
    'iterations': iteration,
    'final_score': ctx.scratch.get('score', 0.0),
    'code_excerpt': ctx.scratch.get('code', '')[:400],
    'docs_excerpt': ctx.scratch.get('docs', '')[:400],
}
summary
